In [2]:
!nvidia-smi

Sat Aug  8 17:45:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!rm -rf /content/sequence-modelling

In [2]:
!git clone https://github.com/Dorcas-Joy-Kahunguka/sequence-modelling
%cd sequence-modelling/mechinterp-typoglycemia/

Cloning into 'sequence-modelling'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 107 (delta 31), reused 95 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 39.75 KiB | 5.68 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/sequence-modelling/mechinterp-typoglycemia


In [3]:
%pip install transformer_lens

In [4]:
import transformer_lens
from transformer_lens import HookedTransformer

import sys
sys.path.append(".")

import threading
from concurrent.futures import ThreadPoolExecutor
from src.data_utils import load_json_dataset

import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

import torch
import pandas as pd
import matplotlib.pyplot as plt





[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [5]:
TOP_K = 5


@torch.no_grad()
def predict_next_word(model, sentence, top_k=TOP_K):
    """
    Run `model` on `sentence` and return the top-k predicted next tokens
    (decoded to strings, stripped) along with their probabilities, for the
    final token position.
    """
    tokens = model.to_tokens(sentence, prepend_bos=True)
    logits = model(tokens)
    last_logits = logits[0, -1]
    probs = torch.softmax(last_logits, dim=-1)
    top_probs, top_ids = probs.topk(top_k)
    words = [model.tokenizer.decode([tid]).strip() for tid in top_ids.tolist()]
    return {
        "top1": words[0],
        "top_k_words": words,
        "top_k_probs": top_probs.tolist(),
    }


def predict_pair(idx):
    """Run clean + scrambled prediction for one pair concurrently on two threads."""
    with ThreadPoolExecutor(max_workers=2) as executor:
        fut_clean = executor.submit(predict_next_word, model_clean, clean_sentences[idx])
        fut_scrambled = executor.submit(predict_next_word, model_scrambled, scrambled_sentences[idx])
        return fut_clean.result(), fut_scrambled.result()

In [6]:
MODEL_NAME = "gpt2-small"  
device = "cuda" if torch.cuda.is_available() else "cpu"

model_clean = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
model_scrambled = HookedTransformer.from_pretrained(MODEL_NAME, device=device)

model_clean.eval()
model_scrambled.eval()
print("Loaded two model instances on", device)

/tmp/ipykernel_11706/3822233280.py:4: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model_clean = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer


/tmp/ipykernel_11706/3822233280.py:5: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model_scrambled = HookedTransformer.from_pretrained(MODEL_NAME, device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer
Loaded two model instances on cuda


In [ ]:
ids, clean_sentences, scrambled_sentences, target_words = load_json_dataset("data/processed/clean.json", "data/processed/scramble.json","data/processed/target.json")

In [ ]:
records = []
for i in range(len(ids)):
    clean_pred, scrambled_pred = predict_pair(i)
    records.append({
        "id": ids[i],
        "clean_sentence": clean_sentences[i],
        "scrambled_sentence": scrambled_sentences[i],
        "true_word": true_words[i],
        "clean_top1": clean_pred["top1"],
        "clean_top_k": clean_pred["top_k_words"],
        "scrambled_top1": scrambled_pred["top1"],
        "scrambled_top_k": scrambled_pred["top_k_words"],
    })
    if (i + 1) % 25 == 0:
        print(f"{i + 1}/{len(ids)} pairs done")

print("Done:", len(records), "pairs predicted.")